<a href="https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/SUYAIBALSIFAT/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 147 (delta 57), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 1.87 MiB | 8.41 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/flyrank-ml-internship


## 1. Method choice and why

I'm using Logistic Regression and Random Forest, per the "yes/no with an observed label" row of the method table — readable model first, stronger model second. My target is a classification/ranking problem (needs_review, 0 or 1), so these fit directly. I'm also reporting permutation-style feature importance from the Random Forest to see what it actually leans on.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

I used a GROUPED split by client_id (not a random row split), because pages from the same client can share patterns (writing style, site structure, industry) that would let the model "cheat" by seeing similar pages from the same client in both train and test. A grouped split by client is the honest test of whether the model generalizes to CLIENTS it has never seen, which is closer to how this would actually be used in production (on a new client).

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["needs_review"] = ((df.trend_direction == "down") & (df.impressions_90d >= 100)).astype(int)

feature_cols = ["search_volume","competition","cpc","word_count","char_count",
    "impressions_90d","clicks_90d","sessions_90d","users_90d","engaged_sessions_90d",
    "ai_sessions_90d","scroll_events_90d","days_with_impressions","days_with_sessions",
    "content_age_days","days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct"]

data = df.dropna(subset=feature_cols + ["client_id"]).copy()
X = data[feature_cols].fillna(0)
y = data["needs_review"]
groups = data["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

overlap = len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
print(f"train rows: {len(X_tr)}, test rows: {len(X_te)}")
print(f"base rate (test): {y_te.mean():.3f}")
print(f"client overlap between train/test: {overlap}")  # must be 0

train rows: 14152, test rows: 5745
base rate (test): 0.491
client overlap between train/test: 0


## 3. Train + compare vs my baseline

Comparing on the SAME test split and SAME metric (precision@50) as my Week-4 baseline.

In [5]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# recompute the baseline rule score from ML-07 on this same data
base_declining = (data.trend_direction == "down") & (data.impressions_90d >= 100)
base_low_ctr = (data.impressions_90d >= 500) & (data.avg_position.between(0, 20, inclusive="right")) & (data.ctr < 0.5)
baseline_score = (data["impressions_90d"] * (base_declining.astype(int) + base_low_ctr.astype(int))).iloc[test_idx].values
p50_base = precision_at_k(baseline_score, y_te.values, 50)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_tr)
X_tr_scaled = scaler.transform(X_tr)
X_te_scaled = scaler.transform(X_te)

log_reg = LogisticRegression(max_iter=2000).fit(X_tr_scaled, y_tr)
lr_probs = log_reg.predict_proba(X_te_scaled)[:, 1]
p50_lr = precision_at_k(lr_probs, y_te.values, 50)
auc_lr = roc_auc_score(y_te, lr_probs)

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
rf_probs = rf.predict_proba(X_te)[:, 1]
p50_rf = precision_at_k(rf_probs, y_te.values, 50)
auc_rf = roc_auc_score(y_te, rf_probs)

results = pd.DataFrame({
    "method": ["baseline rule", "logistic regression", "random forest"],
    "precision@50": [p50_base, p50_lr, p50_rf],
    "roc_auc": [None, round(auc_lr, 3), round(auc_rf, 3)],
})
results

,method,precision@50,roc_auc
0,baseline rule,0.74,NaN
1,logistic regression,0.90,0.815
2,random forest,0.66,0.815


## 4. Errors and interpretation

Surprising result: the baseline rule (precision@50 = 0.74) already performs close to or better than Random Forest (0.66), even though Random Forest has the higher ROC-AUC (0.815 vs no AUC for the rule). Logistic Regression did best at precision@50 (0.88).

Why this happened, most likely: my baseline rule's "declining_with_demand" component is built from trend_direction == "down" AND impressions_90d >= 100 — which is almost exactly my label definition. So the baseline isn't a naive rule here, it's very close to the answer by construction, making it an unusually strong baseline to beat. This is a useful, honest finding: it shows my current label is too easy to reproduce from one obvious rule, and a stronger capstone version should use a FUTURE-window label (e.g., "declines further over the next 30 days") instead of a same-window proxy — this exact problem is flagged in the lane guide.

Random Forest's top features were impressions_90d, days_with_impressions, and avg_position — all plausible (a page's visibility, historical exposure, and ranking position naturally shape whether it's declining), not a suspiciously perfect single feature, so I don't see obvious leakage.

In [4]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.head(6)

,0
impressions_90d,0.193747
days_with_impressions,0.142588
avg_position,0.065164
content_age_days,0.058379
char_count,0.055502
word_count,0.054878


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.